In [3]:
import ee
import geemap
import geopandas as gpd
import numpy as np

In [4]:
ee.Authenticate()

In [5]:
ee.Initialize(project='my-project-01-487803')

In [6]:
m = geemap.Map()
m.add_basemap('HYBRID')
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [7]:
counties = ee.FeatureCollection('TIGER/2018/Counties')
ga = (
    counties
    .filter(ee.Filter.eq('STATEFP', '13'))
    .filter(ee.Filter.inList('NAME', ['Gilmer', 'Fannin', 'Union', 'Towns']))
)
geometry = ga.geometry()

In [8]:
gdf = geemap.ee_to_gdf(ga)
gdf.explore()

In [9]:
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED');

preHelene = ee.Date('2024-09-24')
postHelene = ee.Date('2024-10-04')

csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
qa_band = 'cs_cdf'
clearThreshold = 0.5

def mask_clouds(img):
  return img.updateMask(img.select(qa_band).gte(clearThreshold))

In [10]:
preComposite = (
    s2
    .filter(ee.Filter.date(preHelene.advance(-3, 'month'), preHelene))
    .filterBounds(geometry)
    .linkCollection(csPlus, [qa_band])
    .map(mask_clouds)
    .median()
)

postComposite = (
    s2
    .filter(ee.Filter.date(postHelene, postHelene.advance(3, 'month')))
    .filterBounds(geometry)
    .linkCollection(csPlus, [qa_band])
    .map(mask_clouds)
    .median()
)

print(preComposite.bandNames().getInfo())
print(postComposite.bandNames().getInfo())

['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE', 'cs_cdf']
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE', 'cs_cdf']


In [11]:
def addNDVI(image):
  ndvi = image.normalizedDifference(['B8', 'B4']).rename('ndvi')
  return image.addBands(ndvi)

preNDVI = addNDVI(preComposite).select('ndvi')
postNDVI = addNDVI(postComposite).select('ndvi')

In [16]:
ndviVis = {'min': -0.4, 'max': 0.9, 'palette': 'RdYlGn'}
m.add_layer(preNDVI.clip(geometry), ndviVis, 'Pre-Helene NDVI', False)
m.add_layer(postNDVI.clip(geometry), ndviVis, 'Post-Helene NDVI', False)
m.centerObject(geometry, 10)
m

Map(bottom=26343.0, center=[34.80741140484711, -84.20680032351353], controls=(WidgetControl(options=['position…

In [17]:
ndviDiff = postNDVI.subtract(preNDVI)
diffVis = {
    'min': -0.5,
    'max': 0.5,
    'palette': 'RdYlGn'
}
m.add_layer(ndviDiff.clip(geometry), diffVis, 'Post - Pre Helene Difference', False)

In [18]:
import os
output_folder = 'output'
if not os.path.exists(output_folder):
  os.mkdir(output_folder)
out_geojson = os.path.join(output_folder, 'N_GA_Counties')
geemap.ee_to_geojson(ga, out_geojson)

In [ ]:
geemap.ee_export_image_to_drive(
    preNDVI, description="PreHeleneNDVI", folder="GEE_Exports", region=geometry, scale=10
)

geemap.ee_export_image_to_drive(
    postNDVI, description="PostHeleneNDVI", folder="GEE_Exports", region=geometry, scale=10
)

geemap.ee_export_image_to_drive(
    ndviDiff, description="DiffHeleneNDVI", folder="GEE_Exports", region=geometry, scale=10
)